# 🌾 AgroCrédito Colombia - Generación de Datos

**Equipo:** Andrés, Sebastián, Julián, Yuri - Talento Tech 2

Este cuaderno genera el dataset sintético de créditos rurales colombianos con 2,000 registros y 7 sectores productivos.

In [ ]:
import os
import numpy as np
import pandas as pd

## 1. Configuración del Dataset

In [ ]:
DEPARTAMENTOS = [
    "Antioquia", "Atlántico", "Bolívar", "Boyacá", "Caldas",
    "Cauca", "Cesar", "Córdoba", "Cundinamarca", "Chocó",
    "Huila", "La Guajira", "Magdalena", "Meta", "Nariño",
    "Norte de Santander", "Quindío", "Risaralda", "Santander",
    "Sucre", "Tolima", "Valle del Cauca", "Arauca", "Caquetá",
    "Guainía", "Guaviare", "Putumayo", "Vaupés", "Vichada"
]

SECTORES = [
    "Agricultura", "Ganadería", "Piscicultura", "Porcícola",
    "Avicultura", "Energía_Solar", "Tecnología_Agrícola"
]

PRODUCTOS_POR_SECTOR = {
    "Agricultura": ["Café", "Cacao", "Arroz", "Maíz", "Frutas", "Hortalizas", "Plátano", "Yuca", "Papa", "Cebolla"],
    "Ganadería": ["Leche", "Carne_Bovina", "Carne_Ovina", "Carne_Caprina", "Cuero"],
    "Piscicultura": ["Tilapia", "Trucha", "Camarón", "Pez_Libri", "Pez_Gato"],
    "Porcícola": ["Cerdo_Encaste", "Cerdo_Mestizo", "Lechón", "Chancho_Gordo"],
    "Avicultura": ["Pollo_Parrillero", "Huevo_Posterior", "Pavo", "Codorniz"],
    "Energía_Solar": ["Placa_Fotovoltaica", "Bomba_Solar", "Secador_Solar", "Iluminación_Solar"],
    "Tecnología_Agrícola": ["Riego_Goteo", "Drones_Agrícolas", "Sensores_Humedad", "Automatización_Invernadero"]
}

print(f"Departamentos: {len(DEPARTAMENTOS)}")
print(f"Sectores: {len(SECTORES)}")

## 2. Generación del Dataset

In [ ]:
np.random.seed(42)
n_samples = 2000

# Datos personales
cedulas = [f"{np.random.randint(10000000, 99999999)}" for _ in range(n_samples)]
departamentos = np.random.choice(DEPARTAMENTOS, n_samples)
sectores = np.random.choice(SECTORES, n_samples, p=[0.30, 0.20, 0.15, 0.10, 0.08, 0.07, 0.10])
productos = [np.random.choice(PRODUCTOS_POR_SECTOR[s]) for s in sectores]

# Datos del agricultor
edades = np.random.randint(18, 75, n_samples)
experiencia = np.random.randint(0, 45, n_samples)
hectareas = np.random.uniform(0.5, 50, n_samples).round(2)
ingresos_mensuales = np.random.uniform(800000, 15000000, n_samples).round(0)

# Datos del crédito
monto_prestamo = np.random.uniform(5000000, 200000000, n_samples).round(0)
plazo_meses = np.random.choice([12, 24, 36, 48, 60], n_samples)
garantia = np.random.choice([0, 1], n_samples, p=[0.35, 0.65])
subsidio = np.random.choice([0, 1], n_samples, p=[0.40, 0.60])

# Tecnología
tecnologia_count = np.random.randint(0, 5, n_samples)
tiene_energia_solar = np.random.choice([0, 1], n_samples, p=[0.6, 0.4])
tiene_riego = np.random.choice([0, 1], n_samples, p=[0.5, 0.5])

# Producción
produccion_anual_ton = np.random.uniform(1, 200, n_samples).round(2)
exportaciones = np.random.choice([0, 1], n_samples, p=[0.7, 0.3])

print(f"Registros generados: {n_samples}")

## 3. Fórmula de Tasa de Interés

In [ ]:
# Fórmula generativa de la tasa
tasa_base = 18.0
tasa = (
    tasa_base
    - garantia * 3.2
    - subsidio * 5.5
    - experiencia * 0.06
    - (ingresos_mensuales / 1000000) * 0.3
    - tecnologia_count * 0.5
    - tiene_energia_solar * 1.0
    + (plazo_meses - 12) * 0.03
)
ruido = np.random.normal(0, 0.8, n_samples)
tasa = (tasa + ruido).clip(6.0, 31.0).round(2)

print(f"Tasa promedio: {tasa.mean():.2f}% E.A.")
print(f"Tasa mínima: {tasa.min():.2f}% E.A.")
print(f"Tasa máxima: {tasa.max():.2f}% E.A.")

## 4. Creación del DataFrame

In [ ]:
cedulas_arr = np.array(cedulas, dtype=object)

df = pd.DataFrame({
    'Cedula': cedulas_arr,
    'Departamento': departamentos,
    'Sector_Productivo': sectores,
    'Producto': productos,
    'Edad': edades,
    'Experiencia_Anios': experiencia,
    'Hectareas': hectareas,
    'Ingresos_Mensuales_COP': ingresos_mensuales,
    'Monto_Prestamo_COP': monto_prestamo,
    'Plazo_Meses': plazo_meses,
    'Garantia_Respaldada': garantia,
    'Subsidio_Gobierno': subsidio,
    'Tecnologias_Usadas': tecnologia_count,
    'Tiene_Energia_Solar': tiene_energia_solar,
    'Tiene_Riego': tiene_riego,
    'Produccion_Anual_Ton': produccion_anual_ton,
    'Exporta_Productos': exportaciones,
    'Tasa_Interes_EA': tasa
})

print(f"DataFrame creado: {df.shape[0]} filas, {df.shape[1]} columnas")
df.head(10)

## 5. Introducción de Valores Nulos (para ETL)

In [ ]:
mask_cedula = np.random.rand(n_samples) < 0.02
mask_experiencia = np.random.rand(n_samples) < 0.025
mask_garantia = np.random.rand(n_samples) < 0.015

df.loc[mask_cedula, 'Cedula'] = np.nan
df.loc[mask_experiencia, 'Experiencia_Anios'] = np.nan
df.loc[mask_garantia, 'Garantia_Respaldada'] = np.nan

print(f"Nulos por columna:")
print(df.isnull().sum())

## 6. Guardar Dataset

In [ ]:
os.makedirs('data', exist_ok=True)
df.to_csv('data/creditos_rurales.csv', index=False)
print(f"Dataset guardado: data/creditos_rurales.csv ({len(df)} registros)")

## Resumen

- **2,000 registros** de créditos rurales
- **7 sectores productivos** cubiertos
- **29 departamentos** de Colombia
- Variables con nulos intencionales para practicar ETL
- Tasa de interés calculada con fórmula realista